# ClaimLens — held-out template test

**Question:** did fine-tuning teach the model to *read claim documents*, or just to read *our two training
layouts*?

This notebook generates 100 documents in two **new** templates the model never saw in training: a boxed "Notice
of Loss" form and a "Statement of Account". Fields sit in different places, values sit *below* their labels, and
the wording differs ("Insured" instead of "Policyholder name", "Our reference" instead of "Claim number",
"Balance due" instead of "Total"). It then evaluates **both** the un-tuned baseline and the SFT model on them.

### Setup
- **Settings:** GPU T4 x2, Internet **On**
- **Inputs (two):**
  1. `claimlens-code`, updated to the latest zip
  2. **Your SFT notebook's output**, which holds the trained adapter. Use **Add Input → Your Work → Notebooks**,
     pick the Phase 2 SFT notebook, and choose the version that ran the full training.
- Dry run with `QUICK_TEST = True` (3 documents each, about 10 minutes), then set it to `False` and **Save &
  Run All** (about 1 hour 45 minutes).

In [ ]:
QUICK_TEST = True
UPLOADED = "/kaggle/input"
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
ADAPTER_NAME = "sft_v1"
N_HOLDOUT = 100

In [ ]:
import os, shutil, sys, glob
os.chdir("/kaggle/working")
# the project copy that knows the held-out templates
hits = [p for p in glob.glob(f"{UPLOADED}/**/claimlens/generator.py", recursive=True)
        if "claim_form_b" in open(p, encoding="utf-8").read()]
assert hits, "Held-out code not found -- upload the new zip as a New Version of claimlens-code"
src = os.path.dirname(os.path.dirname(max(hits, key=os.path.getmtime)))
print("Using project at:", src)
shutil.rmtree("claimLens", ignore_errors=True)
shutil.copytree(src, "claimLens")
os.chdir("/kaggle/working/claimLens")
sys.path.insert(0, os.getcwd())

In [ ]:
!pip install -q -U "transformers>=4.49" accelerate "pydantic>=2" peft
!pip uninstall -y -q torchao
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Data: the original splits plus the held-out templates

In [ ]:
!python -m claimlens generate --out data --train 300 --val 50 --test 100 --fair-sets 25 --fair-variants 4 --holdout {N_HOLDOUT} > /dev/null
import json
info = json.load(open("data/dataset_info.json"))
print("test hash still matches Phase 1:", info["split_hashes"]["test"].startswith("10005cb99375adfb"))
print("holdout templates:", info["holdout_templates"], "| docs:", info["counts"]["holdout"])

## 2. Find the trained adapter
It checks the fingerprint against the training card, so we prove this is exactly the model that was trained.

In [ ]:
import zipfile
cfgs = glob.glob(f"{UPLOADED}/**/adapters/{ADAPTER_NAME}/adapter_config.json", recursive=True)
if not cfgs:  # fall back to the packaged zip from the SFT notebook
    zips = glob.glob(f"{UPLOADED}/**/sft_results.zip", recursive=True)
    assert zips, "Adapter not found: add the SFT notebook's output as an input"
    zipfile.ZipFile(zips[0]).extractall("/kaggle/working/sft_unzipped")
    cfgs = glob.glob(f"/kaggle/working/sft_unzipped/**/adapters/{ADAPTER_NAME}/adapter_config.json", recursive=True)
ADAPTER_DIR = os.path.dirname(cfgs[0])
from claimlens import sft
FINGERPRINT = sft.adapter_fingerprint(ADAPTER_DIR)
card = json.load(open(os.path.join(ADAPTER_DIR, "training_card.json")))
print("adapter:", ADAPTER_DIR)
print("fingerprint:", FINGERPRINT, "| matches training card:", FINGERPRINT == card.get("adapter_sha256"))
print("trained on:", card["data"]["train"], "| quick test?", card["training"]["quick_test"])

## 3. Baseline (un-tuned) on the held-out templates

In [ ]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor
from claimlens.predictors import HFVisionPredictor
from claimlens.runner import run_predictions, evaluate_run

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map={"": 0})
model.eval()
BASE_REVISION = getattr(model.config, "_commit_hash", None) or "unknown"
limit = 3 if QUICK_TEST else None
tag = "quick_" if QUICK_TEST else ""

base = HFVisionPredictor.from_model(model, processor, MODEL_ID, BASE_REVISION)
base_dir = run_predictions(base, "data", "holdout", runs_dir="runs", run_id=f"{tag}baseline_v2_greedy_holdout", limit=limit)
evaluate_run(base_dir)
print(open(f"{base_dir}/report.md").read())

## 4. SFT model on the same held-out documents

In [ ]:
from peft import PeftModel
model = PeftModel.from_pretrained(model, ADAPTER_DIR).merge_and_unload()
model.eval()
tuned = HFVisionPredictor.from_model(model, processor, MODEL_ID, f"{BASE_REVISION}+adapter:{ADAPTER_NAME}@{FINGERPRINT}")
sft_dir = run_predictions(tuned, "data", "holdout", runs_dir="runs", run_id=f"{tag}{ADAPTER_NAME}_v2_greedy_holdout", limit=limit)
evaluate_run(sft_dir)
print(open(f"{sft_dir}/report.md").read())

## 5. Side by side

In [ ]:
from claimlens.compare import compare
from IPython.display import Markdown, display
table = compare([base_dir, sft_dir])
open("holdout_comparison.md", "w").write(table)
display(Markdown(table))

In [ ]:
!cd /kaggle/working/claimLens && zip -qr /kaggle/working/holdout_results.zip runs data/holdout data/dataset_info.json holdout_comparison.md && ls -lh /kaggle/working/holdout_results.zip